# Semantic Router — Embedding-Based Query Routing
## A Hands-On Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/59-semantic-router/semantic_router_workbook.ipynb)

When a user sends a query, how do you decide which specialist agent or prompt should handle it — without asking the LLM? **Semantic routing** embeds the query and compares it to pre-embedded example phrases for each route. The route with the highest cosine similarity wins — no LLM call needed for the routing decision itself. By the end of this workshop you will understand *how* cosine similarity enables intent classification, *how* to build a multi-route embedding index, and *how* to wire routing into a LangGraph pipeline.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Semantic routing vs LLM-based intent classification |
| 2 | **Route Index** — Embed example phrases for each route |
| 3 | **Cosine Similarity** — How the matching score works |
| 4 | **Routing in Action** — Score queries against all routes |
| 5 | **Full Pipeline** — route_query → handle with LangGraph |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with `.venv` and `requirements.txt` installed, **or** Google Colab (install cell below)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Thorne, J. et al. (2021). *Zero-Shot Dense Retrieval with Momentum Adversarial Domain Invariant Representations.* https://arxiv.org/abs/2110.07760  
> semantic-router library (Aurelio AI): https://github.com/aurelio-labs/semantic-router

In [ ]:
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain",
        "langchain-openai",
        "langgraph",
        "numpy",
        "python-dotenv",
    ])
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")

---

## Part 1 — Semantic Routing vs LLM-Based Classification

---

### Two ways to classify intent

**Option A — LLM Classification:**
```
Query → LLM("Is this about code, billing, or general?") → route
```
- ✅ Flexible, handles edge cases
- ❌ Costs tokens, adds ~500ms latency per request
- ❌ Requires a prompt, can hallucinate route names

**Option B — Semantic Routing (this workshop):**
```
Query → embed() → cosine_similarity(route_embeddings) → route
```
- ✅ No LLM call for routing — ~10ms after index is built
- ✅ Deterministic and auditable — you can inspect every score
- ✅ Works with any embedding model
- ❌ Requires curated example phrases per route
- ❌ Struggles with truly ambiguous queries

### How it works

```
BUILD PHASE (once at startup):
  For each route (code, billing, general):
    Embed 4-5 representative example phrases
    Store as a list of vectors

ROUTING PHASE (per query):
  1. Embed the incoming query
  2. For each route, compute max cosine_sim(query, example_i)
  3. Route = argmax over all routes
  4. If max_score < threshold → fallback to "general"
```

In [ ]:
import numpy as np

SIMILARITY_THRESHOLD = 0.6

ROUTES = {
    "code": [
        "How do I write a Python function?",
        "Debug my code",
        "What is a list comprehension?",
        "Explain async/await in Python",
        "How do I use pandas DataFrames?",
    ],
    "billing": [
        "How do I upgrade my plan?",
        "What does the pro subscription include?",
        "Can I get a refund?",
        "How much does the API cost?",
        "Where is my invoice?",
    ],
    "general": [
        "What is your company?",
        "Who made this product?",
        "What are your hours?",
        "How do I contact support?",
    ],
}

SAMPLE_QUERIES = [
    "How do I reverse a list in Python?",
    "I need to cancel my subscription",
    "What time do you open?",
    "Can you help me fix a syntax error?",
    "What are your pricing tiers?",
]

print(f"{len(ROUTES)} routes defined:")
for route, phrases in ROUTES.items():
    print(f"  {route}: {len(phrases)} example phrases")
print(f"\n{len(SAMPLE_QUERIES)} test queries ready")

---

## Part 2 — Building the Route Index

---

Embed all example phrases for each route. This happens once at startup — the vectors are reused for every incoming query.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Build: {route_name: [vector, vector, ...]}
route_vecs: dict[str, list[list[float]]] = {
    route: embeddings.embed_documents(phrases)
    for route, phrases in ROUTES.items()
}

dim = len(route_vecs["code"][0])
print(f"Route index built:")
for route, vecs in route_vecs.items():
    print(f"  {route}: {len(vecs)} vectors × {dim} dims")

---

## Part 3 — Cosine Similarity

---

Cosine similarity measures the angle between two vectors, not their magnitude:

```
sim(A, B) = (A · B) / (|A| × |B|)

Range: -1 (opposite) → 0 (orthogonal) → 1 (identical)
```

For semantic routing, the score for a route is the **max** similarity across all its example phrases — if the query matches *any* of the examples, it qualifies for that route.

In [ ]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    a_arr, b_arr = np.array(a), np.array(b)
    return float(np.dot(a_arr, b_arr) / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)))


def score_query(query: str) -> dict[str, float]:
    query_vec = embeddings.embed_query(query)
    return {
        route: max(cosine_similarity(query_vec, v) for v in vecs)
        for route, vecs in route_vecs.items()
    }


# Demo: score one query against all routes
demo_query = "How do I reverse a list in Python?"
scores = score_query(demo_query)

print(f"Query: '{demo_query}'")
print(f"\n{'Route':<10} {'Score':>8}  {'Bar':<30}")
print("-" * 52)
for route, score in sorted(scores.items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 30)
    print(f"{route:<10} {score:>8.4f}  {bar}")
print(f"\nThreshold: {SIMILARITY_THRESHOLD}  →  winner: {max(scores, key=scores.get)}")

---

## Part 4 — Routing All Test Queries

---

Score every test query against all routes and show the routing decision.

In [ ]:
def find_best_route(query: str) -> tuple[str, dict[str, float]]:
    scores = score_query(query)
    best = max(scores, key=lambda r: scores[r])
    if scores[best] < SIMILARITY_THRESHOLD:
        best = "general"
    return best, scores


routes_list = list(ROUTES.keys())
header = f"{'Query':<45} {'Route':<10}" + "".join(f" {r:>8}" for r in routes_list)
print(header)
print("-" * len(header))

for query in SAMPLE_QUERIES:
    best, scores = find_best_route(query)
    score_cols = "".join(f" {scores[r]:>8.4f}" for r in routes_list)
    print(f"{query[:43]:<45} {best:<10}{score_cols}")

---

## Part 5 — Full Pipeline with LangGraph

---

Two nodes:

```
START
  │
  ▼
route_query  ─── embed query, cosine_sim vs route index → route + scores
  │
  ▼
handle       ─── specialist prompt based on route → answer
  │
  ▼
 END
```

Each route uses a different system persona — the LLM only runs for the *answer*, never for routing.

In [ ]:
from typing import TypedDict

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

HANDLER_PROMPTS = {
    "code":    "You are a coding assistant. Answer this programming question concisely:\n{query}",
    "billing": "You are a billing support agent. Answer this account/payment question:\n{query}",
    "general": "You are a general support agent. Answer this question:\n{query}",
}

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


class SemanticRouterState(TypedDict):
    query:  str
    route:  str
    scores: dict
    answer: str


def route_query_node(state: SemanticRouterState) -> dict:
    best, scores = find_best_route(state["query"])
    return {"route": best, "scores": scores}


def handle_node(state: SemanticRouterState) -> dict:
    prompt = HANDLER_PROMPTS[state["route"]].format(query=state["query"])
    return {"answer": llm.invoke([HumanMessage(content=prompt)]).content.strip()}


graph = StateGraph(SemanticRouterState)
graph.add_node("route_query", route_query_node)
graph.add_node("handle", handle_node)
graph.add_edge(START, "route_query")
graph.add_edge("route_query", "handle")
graph.add_edge("handle", END)
app = graph.compile()

print("Pipeline compiled: route_query → handle")

In [ ]:
for query in SAMPLE_QUERIES:
    result: SemanticRouterState = app.invoke(
        {"query": query, "route": "", "scores": {}, "answer": ""}
    )
    print(f"{'=' * 65}")
    print(f"Query  : {result['query']}")
    print(f"Route  : {result['route']}  (scores: {', '.join(f'{k}={v:.3f}' for k,v in result['scores'].items())})")
    print(f"Answer : {result['answer'][:200]}")
    print()

---

### Exercise 1 — Add a New Route

**Goal:** Add a `"technical"` route for questions about APIs and infrastructure:

```python
ROUTES["technical"] = [
    "How do I authenticate with the API?",
    "What is the rate limit?",
    "How do I set environment variables?",
    "What SDK should I use?",
]
```

1. Add the route and rebuild the index
2. Add a handler prompt for the new route
3. Test with: `"How do I get my API key?"` — does it route to `technical` or `billing`?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
# TODO: add ROUTES["technical"] with 4 example phrases
# TODO: rebuild route_vecs
# TODO: add HANDLER_PROMPTS["technical"]
# TODO: test "How do I get my API key?" and print route + scores
pass

### Exercise 2 — Tune the Threshold

**Goal:** Try `SIMILARITY_THRESHOLD = 0.75` and `SIMILARITY_THRESHOLD = 0.5`.

For each threshold, run all 5 test queries and note:
- How many route to `general` (fallback)?
- At threshold `0.5`, does any query get misrouted?
- What threshold gives the best precision/recall trade-off for this corpus?

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
for threshold in [0.50, 0.60, 0.75]:
    # TODO: run all SAMPLE_QUERIES with this threshold, count fallbacks
    pass

---

## What's Next?

Semantic routing is the front door of any multi-agent system — it decides which specialist handles each query without burning LLM tokens.

- **Example 60 — Web Scraper Agent** ([`../60-web-scraper-agent/`](../60-web-scraper-agent/README.md)): after routing, the agent fetches and chunks a live web page for RAG
- **Example 80 — Multi-Agent Supervisor** ([`../80-multi-agent-supervisor/`](../80-multi-agent-supervisor/README.md)): LLM-based routing (compare to this embedding-based approach)
- **semantic-router library**: Aurelio AI's production-grade semantic router with caching, telemetry, and hybrid routing

---

*Workshop complete. Pausing here — confirm to continue to the next notebook.*

---

## Exercise Answer Key

In [ ]:
# Exercise 1 — Add technical route

ROUTES["technical"] = [
    "How do I authenticate with the API?",
    "What is the rate limit?",
    "How do I set environment variables?",
    "What SDK should I use?",
]
HANDLER_PROMPTS["technical"] = "You are a technical integration specialist. Answer this API/infrastructure question:\n{query}"

route_vecs_extended = {
    route: embeddings.embed_documents(phrases)
    for route, phrases in ROUTES.items()
}

test_q = "How do I get my API key?"
query_vec = embeddings.embed_query(test_q)
scores = {
    route: max(cosine_similarity(query_vec, v) for v in vecs)
    for route, vecs in route_vecs_extended.items()
}
best = max(scores, key=scores.get)
print(f"Query: '{test_q}'")
for route, score in sorted(scores.items(), key=lambda x: -x[1]):
    marker = " ← winner" if route == best else ""
    print(f"  {route:<12} {score:.4f}{marker}")

In [ ]:
# Exercise 2 — Threshold tuning

print(f"{'Threshold':<12} {'Fallbacks':>10}  Routes")
print("-" * 55)

for threshold in [0.50, 0.60, 0.75]:
    results = []
    for query in SAMPLE_QUERIES:
        scores = score_query(query)
        best = max(scores, key=scores.get)
        if scores[best] < threshold:
            best = "general (fallback)"
        results.append(best)
    fallbacks = sum(1 for r in results if "fallback" in r)
    route_summary = ", ".join(results)
    print(f"{threshold:<12} {fallbacks:>10}  {route_summary}")